In [1]:
import pyarrow.parquet as pq
import pandas as pd

table = pq.read_table("player_data/February_10/0019c582-574d-4a53-9f77-554519b75b4c_1298e3e2-2776-4038-ba9b-72808b041561.nakama-0")
df = table.to_pandas()

df['event'] = df['event'].apply(
    lambda x: x.decode('utf-8') if isinstance(x, bytes) else x
)

In [2]:
df

,user_id,match_id,map_id,x,y,z,ts,event
0,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-315.353882,125.675491,-2.592606,1970-01-21 11:52:34.537,Position
1,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-317.804230,125.524902,0.661448,1970-01-21 11:52:34.557,Position
2,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-301.990906,120.119232,19.379868,1970-01-21 11:52:34.572,Position
3,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-290.469269,114.962479,35.831268,1970-01-21 11:52:34.577,Position
4,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-283.831940,114.019295,52.365128,1970-01-21 11:52:34.582,Position
...,...,...,...,...,...,...,...,...
60,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-60.327625,103.034363,82.511040,1970-01-21 11:52:34.900,Loot
61,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-60.649342,103.034363,79.278160,1970-01-21 11:52:34.904,Position
62,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-66.733513,103.034363,76.920998,1970-01-21 11:52:34.909,Position
63,0019c582-574d-4a53-9f77-554519b75b4c,1298e3e2-2776-4038-ba9b-72808b041561.nakama-0,AmbroseValley,-64.453148,103.034363,77.230194,1970-01-21 11:52:34.919,BotKilled


In [3]:
print(df['event'].unique())

['Position' 'Loot' 'BotKilled']


In [4]:
MAP_CONFIG = {
    "AmbroseValley": {"scale": 900, "origin_x": -370, "origin_z": -473},
    "GrandRift": {"scale": 581, "origin_x": -290, "origin_z": -290},
    "Lockdown": {"scale": 1000, "origin_x": -500, "origin_z": -500},
}

def world_to_pixel(df, map_id):
    cfg = MAP_CONFIG[map_id]
    
    u = (df['x'] - cfg['origin_x']) / cfg['scale']
    v = (df['z'] - cfg['origin_z']) / cfg['scale']
    
    df['px'] = u * 1024
    df['py'] = (1 - v) * 1024
    
    return df

In [5]:
!python -m venv venv
!venv\Scripts\activate
!streamlit run app.py

^C


In [ ]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ── Coordinate utils (inline — no import needed) ──────────────────
MINIMAP_SIZE = 1024
MAP_CONFIG = {
    "AmbroseValley": {"scale": 900,  "origin_x": -370, "origin_z": -473},
    "GrandRift":     {"scale": 581,  "origin_x": -290, "origin_z": -290},
    "Lockdown":      {"scale": 1000, "origin_x": -500, "origin_z": -500},
}

def world_to_pixel_vectorized(x_series, z_series, map_id):
    cfg     = MAP_CONFIG[map_id]
    u       = (x_series - cfg["origin_x"]) / cfg["scale"]
    v       = (z_series - cfg["origin_z"]) / cfg["scale"]
    pixel_x = u * MINIMAP_SIZE
    plot_y  = MINIMAP_SIZE - (1 - v) * MINIMAP_SIZE  # Plotly: y=0 at bottom
    return pixel_x.values, plot_y.values

# ── Load ──────────────────────────────────────────────────────────
DATA_DIR = "player_data"   # change path if needed

DAY_LABELS = {
    "February_10": "Feb 10",
    "February_11": "Feb 11",
    "February_12": "Feb 12",
    "February_13": "Feb 13",
    "February_14": "Feb 14",
}

frames, errors = [], 0

for folder_name, date_label in DAY_LABELS.items():
    folder_path = os.path.join(DATA_DIR, folder_name)
    if not os.path.exists(folder_path):
        print(f"  MISSING: {folder_path}")
        continue
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            df_file = pq.read_table(fpath).to_pandas()
            df_file["date"] = date_label
            frames.append(df_file)
        except Exception as e:
            errors += 1

df_all = pd.concat(frames, ignore_index=True)

# ── Preprocess ────────────────────────────────────────────────────
df_all["event"]    = df_all["event"].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else str(x))
df_all["user_id"]  = df_all["user_id"].astype(str)
df_all["is_human"] = ~df_all["user_id"].str.match(r"^\d+$")
df_all["match_id_clean"] = df_all["match_id"].astype(str).str.replace(r"\.nakama-0$", "", regex=True)

# Timestamps — unit is SECONDS (README says ms but it's wrong)
df_all["ts_ns"]       = df_all["ts"].astype(np.int64)            # raw unix seconds as int
df_all["ts_relative"] = df_all.groupby("match_id")["ts_ns"].transform(lambda x: x - x.min())  # seconds from match start

# Pixel coords
df_all["pixel_x"] = np.nan
df_all["plot_y"]  = np.nan
for map_id in df_all["map_id"].unique():
    if map_id not in MAP_CONFIG:
        continue
    mask = df_all["map_id"] == map_id
    px, py = world_to_pixel_vectorized(df_all.loc[mask, "x"], df_all.loc[mask, "z"], map_id)
    df_all.loc[mask, "pixel_x"] = px
    df_all.loc[mask, "plot_y"]  = py

# # Drop duplicates
# before = len(df_all)
# df_all = df_all.drop_duplicates()
# print(f"Loaded {before:,} rows → dropped {before - len(df_all):,} duplicates → {len(df_all):,} clean rows")
print(f"Files failed to parse: {errors}")
print(df_all["event"].value_counts())

In [ ]:
df_all['is_bot'] = ~df_all['is_human']

In [ ]:
df_all[df_all['match_id'] == '039d0edf-6a9f-4dee-9d90-b2f1f8c70efa.nakama-0']['ts_ns'].min()

In [ ]:
df_all[df_all['user_id'] == '1440']['event'].unique()

In [ ]:
"""
LILA BLACK — Full Data Sanity Check
====================================
Run each BLOCK independently in Jupyter.
Assumes df_all is already loaded (from data_loader.py or your own load).
"""

import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# SHARED CONFIG  — edit once, used everywhere
# ─────────────────────────────────────────────────────────────
MATCH_ID  = "039d0edf-6a9f-4dee-9d90-b2f1f8c70efa.nakama-0"
USER_ID   = "0019c582-574d-4a53-9f77-554519b75b4c"
df        = df_all          # your loaded dataframe

# ══════════════════════════════════════════════════════════════
# BLOCK 0 — TIMESTAMP TRUTH CHECK
# (README says ms, you found it's actually seconds — confirm here)
# ══════════════════════════════════════════════════════════════
print("=" * 70)
print("BLOCK 0 — TIMESTAMP INVESTIGATION")
print("=" * 70)

raw_ts = df["ts_ns"].dropna()
sample_val = raw_ts.iloc[0]
print(f"\nRaw sample ts value     : {sample_val}")
print(f"Interpreted as ms       : {pd.to_datetime(sample_val, unit='ms')}")
print(f"Interpreted as s        : {pd.to_datetime(sample_val, unit='s')}")
print(f"\nMin ts                  : {raw_ts.min()}")
print(f"Max ts                  : {raw_ts.max()}")
print(f"Range (if s)            : {(raw_ts.max() - raw_ts.min()) / 60:.1f} minutes")
print(f"Range (if ms)           : {(raw_ts.max() - raw_ts.min()) / 1000:.1f} seconds")

# Per-match duration using 's' interpretation
match_durations = (
    df.groupby("match_id")["ts_ns"]
    .agg(lambda x: (x.max() - x.min()) / 60)
    .describe()
)
print(f"\nPer-match duration stats (unit=s → converted to minutes):")
print(match_durations.round(2))

# VERDICT
median_dur = df.groupby("match_id")["ts_ns"].agg(lambda x: (x.max() - x.min()) / 60).median()
print(f"\n>>> VERDICT: Median match duration = {median_dur:.1f} min → unit is almost certainly SECONDS <<<")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 1 — df_all GLOBAL OVERVIEW
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("BLOCK 1 — GLOBAL df_all OVERVIEW")
print("=" * 70)

total_rows    = len(df)
total_matches = df["match_id"].nunique()
total_users   = df["user_id"].nunique()
human_mask    = df["is_human"]
bot_mask      = ~df["is_human"]

print(f"\nTotal rows (events)     : {total_rows:,}")
print(f"Total matches           : {total_matches:,}")
print(f"Total unique user_ids   : {total_users:,}")
print(f"  → Humans (UUID)       : {df[human_mask]['user_id'].nunique():,}")
print(f"  → Bots  (numeric)     : {df[bot_mask]['user_id'].nunique():,}")
print(f"\nDate range              :")
print(df["date"].value_counts().sort_index())

print(f"\nMap distribution:")
print(df["map_id"].value_counts())

print(f"\nEvent distribution:")
print(df["event"].value_counts())

print(f"\nNull counts per column:")
print(df.isnull().sum())

print(f"\nData types:")
print(df.dtypes)

# Duplicates — full row duplicates
dup_count = df.duplicated().sum()
print(f"\nFull duplicate rows     : {dup_count:,}")
# Duplicates by key fields
dup_key = df.duplicated(subset=["user_id", "match_id", "ts_ns", "event"]).sum()
print(f"Duplicates (user+match+ts+event): {dup_key:,}")
# If duplicates exist — show a sample
if dup_count > 0:
    dup_sample = df[df.duplicated(keep=False)].sort_values(["user_id","match_id","ts_ns"])
    print(f"\nSample duplicate rows (first 6):")
    print(dup_sample.head(6))
    print(f"\nAre duplicates only Position events?")
    print(df[df.duplicated(keep=False)]["event"].value_counts())

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 2 — MATCH-LEVEL STATS
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("BLOCK 2 — PER-MATCH COMPOSITION & COMBAT STATS")
print("=" * 70)

# Build per-match summary
def match_summary(df):
    grp = df.groupby("match_id")
    ev  = df.pivot_table(index="match_id", columns="event", aggfunc="size", fill_value=0)
    
    # User counts
    humans_per_match = df[df["is_human"]].groupby("match_id")["user_id"].nunique().rename("n_humans")
    bots_per_match   = df[~df["is_human"]].groupby("match_id")["user_id"].nunique().rename("n_bots")
    total_events     = grp.size().rename("total_events")
    duration_min     = grp["ts_ns"].agg(lambda x: (x.max()-x.min())/60).rename("duration_min")
    
    summary = pd.concat([humans_per_match, bots_per_match, total_events, duration_min], axis=1).fillna(0)
    
    # Merge event columns safely
    for col in ["Kill","Killed","BotKill","BotKilled","KilledByStorm","Loot","Position","BotPosition"]:
        summary[col] = ev[col] if col in ev.columns else 0
    
    summary["total_players"] = summary["n_humans"] + summary["n_bots"]
    summary["kill_to_storm_ratio"] = (summary["Kill"] / (summary["KilledByStorm"] + 1)).round(2)
    return summary.reset_index()

ms = match_summary(df)

print(f"\nMatches total           : {len(ms):,}")
print(f"\nPlayers per match:")
print(ms[["n_humans","n_bots","total_players"]].describe().round(1))

print(f"\nCombat events per match:")
print(ms[["Kill","Killed","BotKill","BotKilled","KilledByStorm","Loot"]].describe().round(1))

print(f"\nDuration (minutes) per match:")
print(ms["duration_min"].describe().round(2))

# ── ANOMALY: Matches with only humans / only bots ──────────────
only_humans = ms[ms["n_bots"] == 0]
only_bots   = ms[ms["n_humans"] == 0]
mixed       = ms[(ms["n_humans"] > 0) & (ms["n_bots"] > 0)]

print(f"\n{'─'*50}")
print(f"Match composition split:")
print(f"  Only human players    : {len(only_humans)} matches")
print(f"  Only bots             : {len(only_bots)} matches")
print(f"  Mixed (humans+bots)   : {len(mixed)} matches")

# ── ANOMALY: 1 human, 0 bots but has BotKills ──────────────────
weird = ms[(ms["n_humans"] >= 1) & (ms["n_bots"] == 0) & (ms["BotKill"] > 0)]
print(f"\n{'─'*50}")
print(f"ANOMALY — Matches with BotKill events but 0 bots recorded: {len(weird)}")
if not weird.empty:
    print(weird[["match_id","n_humans","n_bots","BotKill","BotKilled","Kill","total_events"]].head(10))
    print("""
  >> Likely cause: Bot files may be MISSING from player_data/ (only human files loaded)
     but the human player's file still logs 'BotKill' when they kill a bot.
     The bot's own file simply wasn't included / loaded correctly.
     This is a DATA COMPLETENESS issue, not a game bug.
    """)

# ── Are people actually fighting each other or dying to storm? ──
print(f"\n{'─'*50}")
print("COMBAT vs STORM DEATHS — Are players meeting each other?")
total_pvp_kills   = ms["Kill"].sum()
total_bot_kills   = ms["BotKill"].sum()
total_storm_deaths = ms["KilledByStorm"].sum()
total_killed       = ms["Killed"].sum()
print(f"  Total H→H kills       : {total_pvp_kills:,}")
print(f"  Total bot kills       : {total_bot_kills:,}")
print(f"  Total storm deaths    : {total_storm_deaths:,}")
print(f"  Total H→H deaths      : {total_killed:,}")
print(f"  Storm % of all deaths : {total_storm_deaths/(total_storm_deaths+total_killed+1)*100:.1f}%")
print(f"  Kill:Storm ratio      : {total_pvp_kills/(total_storm_deaths+1):.2f}x")

pct_0_pvp = (ms["Kill"] == 0).mean() * 100
print(f"\n  Matches with ZERO H→H kills : {pct_0_pvp:.1f}% of all matches")
print(f"  Matches with ZERO storm deaths: {(ms['KilledByStorm']==0).mean()*100:.1f}%")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 7 — ECOSYSTEM HEALTH: BOTS, PLAYER COUNT, STORM, ENGAGEMENT
# ══════════════════════════════════════════════════════════════
print("=" * 70)
print("BLOCK 7 — ECOSYSTEM HEALTH CHECKS")
print("=" * 70)

ms = match_summary(df)  # reuse from Block 2

# ─────────────────────────────────────────────────────────────
# 7A — PLAYER COUNT PER MATCH
# Is there even enough humans per match for PvP to happen?
# ─────────────────────────────────────────────────────────────
print("\n── 7A: HUMAN COUNT PER MATCH ──")
hpm = ms["n_humans"].value_counts().sort_index()
print(hpm.to_string())
solo_matches = (ms["n_humans"] == 1).sum()
print(f"\nMatches with EXACTLY 1 human : {solo_matches} ({solo_matches/len(ms)*100:.1f}%)")
print(f"Matches with 2+ humans       : {(ms['n_humans'] >= 2).sum()} ({(ms['n_humans'] >= 2).mean()*100:.1f}%)")
print(f"Matches with 5+ humans       : {(ms['n_humans'] >= 5).sum()}")
print(f"Matches with 10+ humans      : {(ms['n_humans'] >= 10).sum()}")
print("""
>> If 99.6% matches have 0 H→H kills AND most matches have 1 human —
   that's expected. You can't get H→H kills with 1 player.
   PvP can only happen in the 2+ human matches. Check kill rate THERE.
""")

# Kill rate only in matches where PvP was possible
pvp_possible = ms[ms["n_humans"] >= 2]
pvp_kill_rate = (pvp_possible["Kill"] > 0).mean() * 100
print(f"Of {len(pvp_possible)} matches with 2+ humans:")
print(f"  % that had at least 1 H→H kill : {pvp_kill_rate:.1f}%")
print(f"  Avg kills per match            : {pvp_possible['Kill'].mean():.2f}")

# ─────────────────────────────────────────────────────────────
# 7B — BOT BEHAVIOR ANALYSIS (through human files only)
# Bots missing their own files — what CAN we infer?
# ─────────────────────────────────────────────────────────────
print("\n── 7B: BOT BEHAVIOR (inferred from human event logs) ──")

bot_kills_by_humans  = (df["event"] == "BotKill").sum()      # human killed a bot
bots_killed_humans   = (df["event"] == "BotKilled").sum()    # bot killed a human

print(f"BotKill  (human killed bot)  : {bot_kills_by_humans:,}")
print(f"BotKilled (bot killed human) : {bots_killed_humans:,}")
print(f"Bot lethality ratio          : {bots_killed_humans/(bot_kills_by_humans+1):.3f}  (BotKilled / BotKill)")
print(f"\nBots killed humans {bots_killed_humans}x vs humans killed bots {bot_kills_by_humans}x")

# Bot kill events with actual BotPosition files loaded
bot_position_matches = df[df["event"] == "BotPosition"]["match_id"].nunique()
botkill_matches      = df[df["event"] == "BotKill"]["match_id"].nunique()
print(f"\nMatches where bot MOVEMENT was recorded (BotPosition files loaded): {bot_position_matches}")
print(f"Matches where BotKill happened                                     : {botkill_matches}")
print(f"Matches with BotKill but NO BotPosition files                      : {botkill_matches - bot_position_matches}")
print("""
>> Bot files appear partially loaded. Bot files only contain BotPosition events.
   They don't log: BotKill (bot killing human), BotKilled (bot being killed),
   storm deaths, or loot. Those events only appear in the HUMAN's file.
   This means: bot movement paths are only visible where their files loaded.
   Bot combat behavior is inferred entirely from human-side logs.
""")

# ─────────────────────────────────────────────────────────────
# 7C — STORM ANALYSIS
# Is the storm working? Too slow? Players extracting before it hits?
# ─────────────────────────────────────────────────────────────
print("\n── 7C: STORM ANALYSIS ──")

storm_df = df[df["event"] == "KilledByStorm"]
print(f"Total storm deaths            : {len(storm_df)}")
print(f"Matches with any storm death  : {storm_df['match_id'].nunique()} / {df['match_id'].nunique()}")
print(f"Maps with storm deaths:")
print(storm_df["map_id"].value_counts())

# When in the match do storm deaths happen?
if not storm_df.empty:
    print(f"\nStorm death timing (ts_relative in seconds):")
    print(storm_df["ts_relative"].describe().round(1))
    print(f"\nDo storm deaths cluster at end of match?")
    # Compare storm death time vs match duration
    match_end = df.groupby("match_id")["ts_relative"].max().rename("match_duration")
    storm_timing = storm_df.merge(match_end, on="match_id")
    storm_timing["pct_through_match"] = storm_timing["ts_relative"] / storm_timing["match_duration"]
    print(storm_timing["pct_through_match"].describe().round(2))
    print("""
>> pct_through_match: if median ~0.8-1.0 → storm hits late, players extract first.
   If median ~0.3-0.5 → storm is aggressive, killing players mid-match.
""")

# ─────────────────────────────────────────────────────────────
# 7D — MATCH DURATION vs DEATH TYPE
# Are matches ending due to combat, storm, or extraction?
# ─────────────────────────────────────────────────────────────
print("\n── 7D: HOW ARE MATCHES ENDING? ──")

death_events = ["Killed", "KilledByStorm", "BotKilled"]
deaths_df = df[df["event"].isin(death_events)]

# Per match: last event type
last_events = df.sort_values("ts_relative").groupby("match_id").last()["event"]
print(f"\nMost common LAST event in a match:")
print(last_events.value_counts().head(10))

# Survival rate: humans who never died
humans_who_died = df[df["event"].isin(["Killed","KilledByStorm","BotKilled"])]["user_id"].unique()
all_humans = df[df["is_human"]]["user_id"].unique()
survived = set(all_humans) - set(humans_who_died)
# Note: this is per user across all matches, not per match
print(f"\nUnique humans who NEVER died (across all data) : {len(survived)}")
print(f"Unique humans who died at least once           : {len(set(all_humans) & set(humans_who_died))}")

# Per-match survival
def did_survive(group):
    return not group["event"].isin(death_events).any()

survival_per_match = (
    df[df["is_human"]]
    .groupby(["match_id", "user_id"])
    .apply(did_survive)
    .reset_index(name="survived")
)
survival_rate = survival_per_match["survived"].mean() * 100
print(f"\nPer human-match survival rate : {survival_rate:.1f}%")
print("""
>> High survival rate (>70%) in an extraction shooter = players are extracting
   successfully, OR dying to bots (BotKilled) which doesn't show in "died" check above.
   Low survival = storm/bots killing everyone before extract.
""")

# ─────────────────────────────────────────────────────────────
# 7E — LOOT BEHAVIOUR
# Are players even looting? Loot density vs match duration?
# ─────────────────────────────────────────────────────────────
print("\n── 7E: LOOT BEHAVIOUR ──")

loot_df = df[df["event"] == "Loot"]
print(f"Total loot events             : {len(loot_df):,}")
print(f"Matches with any looting      : {loot_df['match_id'].nunique()}")
print(f"Humans who never looted       : {df[df['is_human']]['user_id'].nunique() - loot_df['user_id'].nunique()}")

loot_per_human_match = (
    loot_df.groupby(["match_id","user_id"]).size().reset_index(name="loots")
)
print(f"\nLoots per human per match:")
print(loot_per_human_match["loots"].describe().round(1))

# When do they loot? Early vs late match
if not loot_df.empty:
    match_end2 = df.groupby("match_id")["ts_relative"].max().rename("match_duration")
    loot_timing = loot_df.merge(match_end2, on="match_id")
    loot_timing["pct"] = loot_timing["ts_relative"] / loot_timing["match_duration"]
    print(f"\nLoot timing (% through match):")
    print(loot_timing["pct"].describe().round(2))
    print(">> Early looting (pct ~0.1-0.3) = players loot before fighting (healthy)")
    print(">> Late looting = players fighting/dying before they can loot (map density issue?)")

# ─────────────────────────────────────────────────────────────
# 7F — IS THIS TEST DATA OR PRODUCTION?
# Signal checks: low pop, no PvP, partial files = test environment
# ─────────────────────────────────────────────────────────────
print("\n── 7F: TEST vs PRODUCTION SIGNAL CHECK ──")

avg_humans   = ms["n_humans"].mean()
avg_bots_loaded = ms["n_bots"].mean()
bot_file_coverage = bot_position_matches / df["match_id"].nunique() * 100
pvp_rate     = (ms["Kill"] > 0).mean() * 100
solo_pct     = (ms["n_humans"] == 1).mean() * 100

signals = {
    f"Avg humans per match          : {avg_humans:.1f}":
        "🔴 Very low" if avg_humans < 3 else "🟡 Low" if avg_humans < 8 else "🟢 Normal",
    f"Solo-human matches            : {solo_pct:.1f}%":
        "🔴 Test signal" if solo_pct > 70 else "🟢 OK",
    f"Bot file coverage             : {bot_file_coverage:.1f}%":
        "🔴 Bot files missing" if bot_file_coverage < 50 else "🟢 OK",
    f"Matches with any PvP kill     : {pvp_rate:.1f}%":
        "🔴 No PvP" if pvp_rate < 5 else "🟡 Low" if pvp_rate < 20 else "🟢 Normal",
    f"Storm death rate              : {(ms['KilledByStorm']>0).mean()*100:.1f}% of matches":
        "🔴 Storm inactive" if (ms['KilledByStorm']>0).mean() < 0.1 else "🟢 OK",
}

for label, verdict in signals.items():
    print(f"  {label}  →  {verdict}")

print("""
OVERALL READ:
  If most signals are 🔴 — this is early internal test data, NOT production traffic.
  Small team testing the game loop: 1 dev per match, bots partially stubbed,
  storm tuning not finalized. The 5 days may be pre-launch QA, not live player data.
  
  This is actually a useful insight for INSIGHTS.md:
  "Data suggests pre-launch testing phase — PvP balance and storm timing
   cannot be evaluated from this dataset at production scale."
""")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 3 — SPECIFIC MATCH DEEP DIVE
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"BLOCK 3 — MATCH DEEP DIVE: {MATCH_ID}")
print("=" * 70)

mdf = df[df["match_id"] == MATCH_ID].copy()
print(f"\nTotal events            : {len(mdf):,}")
print(f"Unique users            : {mdf['user_id'].nunique()}")
print(f"  Humans                : {mdf[mdf['is_human']]['user_id'].nunique()}")
print(f"  Bots                  : {mdf[~mdf['is_human']]['user_id'].nunique()}")
print(f"\nEvent distribution:")
print(mdf["event"].value_counts())

t_min = mdf["ts_ns"].min()
t_max = mdf["ts_ns"].max()
duration_s = t_max - t_min
print(f"\nMatch start (unix s)    : {t_min}")
print(f"Match end   (unix s)    : {t_max}")
print(f"Duration                : {duration_s:.0f}s = {duration_s/60:.2f} min")

print(f"\nEvents per user (sorted):")
print(mdf.groupby("user_id").size().sort_values(ascending=False).to_string())

print(f"\nEvents per user by type:")
print(mdf.groupby(["user_id","event"]).size().unstack(fill_value=0).to_string())

# ── Events AFTER death ─────────────────────────────────────────
print(f"\n{'─'*50}")
print("GHOST CHECK — Events recorded AFTER a player's death event")

death_events = ["Killed", "KilledByStorm", "BotKilled"]
death_ts = (
    mdf[mdf["event"].isin(death_events)]
    .groupby("user_id")["ts_ns"]
    .min()
    .rename("death_ts")
)

ghost_counts = []
for uid, death_time in death_ts.items():
    after_death = mdf[(mdf["user_id"] == uid) & (mdf["ts_ns"] > death_time)]
    if len(after_death) > 0:
        ghost_counts.append({
            "user_id": uid,
            "death_ts": death_time,
            "events_after_death": len(after_death),
            "event_types_after": after_death["event"].value_counts().to_dict()
        })

if ghost_counts:
    print(f"Players with events after death: {len(ghost_counts)}")
    for g in ghost_counts[:5]:
        print(f"  user: {g['user_id'][:20]}... | events after death: {g['events_after_death']} | types: {g['event_types_after']}")
    print("\n  >> Normal in extraction shooters — position events may be buffered/flushed after death.")
    print("     Or spectator mode still logs Position. Not necessarily a bug.")
else:
    print("  Clean — no events recorded after death in this match.")

# ── Coordinate sanity ──────────────────────────────────────────
print(f"\n{'─'*50}")
print("COORDINATE SANITY CHECK (world coords)")
print(f"  x range: [{mdf['x'].min():.1f}, {mdf['x'].max():.1f}]")
print(f"  z range: [{mdf['z'].min():.1f}, {mdf['z'].max():.1f}]")
print(f"  y range: [{mdf['y'].min():.1f}, {mdf['y'].max():.1f}]  (elevation)")

# Map config
MAP_CONFIG = {
    "AmbroseValley": {"scale": 900,  "origin_x": -370, "origin_z": -473},
    "GrandRift":     {"scale": 581,  "origin_x": -290, "origin_z": -290},
    "Lockdown":      {"scale": 1000, "origin_x": -500, "origin_z": -500},
}
map_id = mdf["map_id"].iloc[0]
cfg = MAP_CONFIG.get(map_id, None)
if cfg:
    u_min = (mdf["x"].min() - cfg["origin_x"]) / cfg["scale"]
    u_max = (mdf["x"].max() - cfg["origin_x"]) / cfg["scale"]
    v_min = (mdf["z"].min() - cfg["origin_z"]) / cfg["scale"]
    v_max = (mdf["z"].max() - cfg["origin_z"]) / cfg["scale"]
    print(f"\n  Map: {map_id}")
    print(f"  UV x range (should be 0-1): [{u_min:.3f}, {u_max:.3f}]")
    print(f"  UV z range (should be 0-1): [{v_min:.3f}, {v_max:.3f}]")
    if u_min < -0.1 or u_max > 1.1 or v_min < -0.1 or v_max > 1.1:
        print("  ⚠️  Some coordinates OUT OF BOUNDS — events outside minimap area")
    else:
        print("  ✅  All coords within minimap bounds")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 4 — SPECIFIC USER DEEP DIVE
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print(f"BLOCK 4 — USER DEEP DIVE: {USER_ID}")
print("=" * 70)

udf = df[df["user_id"] == USER_ID].copy()
print(f"\nTotal events            : {len(udf):,}")
print(f"Unique matches played   : {udf['match_id'].nunique()}")
print(f"Matches:")
print(udf["match_id"].unique())
print(f"\nEvent distribution:")
print(udf["event"].value_counts())
print(f"\nMaps played on:")
print(udf["map_id"].value_counts())
print(f"\nDates played:")
print(udf["date"].value_counts().sort_index())

print(f"\nPer-match breakdown for this user:")
per_match = udf.groupby("match_id")["event"].value_counts().unstack(fill_value=0)
print(per_match.to_string())

# Kill/Death ratio
total_kills  = (udf["event"] == "Kill").sum()
total_deaths = (udf["event"].isin(["Killed","KilledByStorm","BotKilled"])).sum()
total_loots  = (udf["event"] == "Loot").sum()
print(f"\nCareer stats:")
print(f"  H→H Kills             : {total_kills}")
print(f"  Deaths (all types)    : {total_deaths}")
print(f"  K/D ratio             : {total_kills/(total_deaths or 1):.2f}")
print(f"  Loots                 : {total_loots}")

# Per match K/D
print(f"\nPer-match K/D:")
for mid, mdf_u in udf.groupby("match_id"):
    k = (mdf_u["event"] == "Kill").sum()
    d = (mdf_u["event"].isin(["Killed","KilledByStorm","BotKilled"])).sum()
    bk = (mdf_u["event"] == "BotKill").sum()
    l  = (mdf_u["event"] == "Loot").sum()
    dur = (mdf_u["ts_ns"].max() - mdf_u["ts_ns"].min()) / 60
    print(f"  {mid[:20]}... K:{k} D:{d} BotK:{bk} Loot:{l} dur:{dur:.1f}min")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 5 — DUPLICATE INVESTIGATION
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("BLOCK 5 — DUPLICATE ROW DEEP DIVE")
print("=" * 70)

dup_mask  = df.duplicated(keep=False)
dup_df    = df[dup_mask]
full_dups = df.duplicated().sum()

print(f"\nFull exact duplicate rows       : {full_dups:,}")
print(f"Rows involved in any duplicate  : {len(dup_df):,}")

if full_dups > 0:
    print(f"\nEvent types in duplicates:")
    print(dup_df["event"].value_counts())
    print(f"\nMaps in duplicates:")
    print(dup_df["map_id"].value_counts())
    print(f"\nDates in duplicates:")
    print(dup_df["date"].value_counts())
    
    # Are they all position events? (most likely)
    pos_dup_pct = (dup_df["event"].isin(["Position","BotPosition"])).mean()*100
    print(f"\nPosition events = {pos_dup_pct:.1f}% of duplicates")
    print("""
  >> LIKELY CAUSE: Parquet files may overlap across days (same match spans midnight
     or data was ingested twice). Check if same (user_id, match_id, ts_ns, event)
     appears in files from DIFFERENT date folders.
    """)
    
    # Check if dupes span multiple dates
    if "date" in df.columns:
        key_cols = ["user_id","match_id","ts_ns","event"]
        cross_date = (
            df.groupby(key_cols)["date"]
            .nunique()
            .reset_index(name="n_dates")
        )
        cross = cross_date[cross_date["n_dates"] > 1]
        print(f"Rows with same key in MULTIPLE date folders: {len(cross):,}")
        if not cross.empty:
            print("  >> These are REAL cross-day duplicates — same event ingested twice.")
            print("  >> Safe to drop_duplicates() after confirming.")

In [ ]:
# ══════════════════════════════════════════════════════════════
# BLOCK 6 — DATA QUALITY RED FLAGS SUMMARY
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("BLOCK 6 — DATA QUALITY RED FLAGS SUMMARY")
print("=" * 70)

issues = []

# 1. Duplicate rows
if df.duplicated().sum() > 0:
    issues.append(f"❗ {df.duplicated().sum():,} exact duplicate rows — likely same file loaded from 2 date folders")

# 2. Out of bounds coordinates
for map_id, cfg in MAP_CONFIG.items():
    mmap = df[df["map_id"] == map_id]
    u = (mmap["x"] - cfg["origin_x"]) / cfg["scale"]
    v = (mmap["z"] - cfg["origin_z"]) / cfg["scale"]
    oob = ((u < -0.1) | (u > 1.1) | (v < -0.1) | (v > 1.1)).sum()
    if oob > 0:
        issues.append(f"❗ {oob:,} out-of-bounds coordinates on {map_id}")

# 3. Matches with BotKills but 0 bots
if len(weird) > 0:
    issues.append(f"❗ {len(weird)} matches have BotKill events but no bot files loaded — incomplete data")

# 4. Very short matches (< 1 min)
short = ms[ms["duration_min"] < 1.0]
if len(short) > 0:
    issues.append(f"⚠️  {len(short)} matches < 1 minute duration — may be abandoned/crashed sessions")

# 5. Matches with 0 events of certain types
zero_pos = ms[(ms.get("Position", 0) == 0) if "Position" in ms.columns else ms["total_events"] < 5]
if len(zero_pos) > 0:
    issues.append(f"⚠️  {len(zero_pos)} matches with very few total events — partial loads?")

# 6. Users with only 1 event total (useless for analysis)
thin_users = df.groupby("user_id").size()
thin = (thin_users == 1).sum()
if thin > 0:
    issues.append(f"⚠️  {thin} users with only 1 event total — noise")

# 7. Ghost events (after death)
all_deaths = df[df["event"].isin(["Killed","KilledByStorm","BotKilled"])].groupby(["user_id","match_id"])["ts_ns"].min().rename("death_ts").reset_index()
all_post = df.merge(all_deaths, on=["user_id","match_id"], how="left")
ghost_total = (all_post["ts_ns"] > all_post["death_ts"]).sum()
if ghost_total > 0:
    issues.append(f"ℹ️  {ghost_total:,} events logged after player's death event — normal (buffered positions/spectator)")

if issues:
    for i in issues:
        print(f"  {i}")
else:
    print("  ✅ No major issues found.")

print(f"\n{'═'*70}")
print("DONE. Look at BLOCK 6 issues list first — those are your data gotchas.")
print(f"{'═'*70}")

In [ ]:
# import pandas as pd

# # ============================================================
# # CONFIG — swap these with your actual df/column names
# # ============================================================
# MATCH_ID = "039d0edf-6a9f-4dee-9d90-b2f1f8c70efa.nakama-0"
# USER_ID  = "0019c582-574d-4a53-9f77-554519b75b4c"

# df = df_all          # your main events dataframe
# match_col   = "match_id"
# user_col    = "user_id"
# event_col   = "event"       # or "event_type"
# ts_col      = "ts_ns"        # or "created_at"
# is_bot_col  = "is_bot"           # set None if you don't have this
# session_col = "None"       # set None if not applicable

# # ============================================================
# # BLOCK 1 — MATCH SANITY CHECK
# # ============================================================
# match_df = df[df[match_col] == MATCH_ID]

# print("=" * 60)
# print(f"MATCH SANITY CHECK: {MATCH_ID}")
# print("=" * 60)

# print(f"\n[1] Total rows (events) in match : {len(match_df)}")
# print(f"[2] Unique users                 : {match_df[user_col].nunique()}")
# print(f"[3] User IDs in match:")
# print(match_df[user_col].unique())

# if is_bot_col and is_bot_col in match_df.columns:
#     bot_counts = match_df.groupby(user_col)[is_bot_col].first()
#     n_bots     = bot_counts.sum()
#     n_humans   = (~bot_counts.astype(bool)).sum()
#     print(f"\n[4] Humans                       : {n_humans}")
#     print(f"[5] Bots                         : {n_bots}")
# else:
#     print("\n[4/5] Bot column not found — skipping bot/human split")

# if event_col in match_df.columns:
#     print(f"\n[6] Unique event types           : {match_df[event_col].nunique()}")
#     print("\n[7] Event type distribution:")
#     print(match_df[event_col].value_counts())

# if ts_col in match_df.columns:
#     t_min =  pd.to_datetime(match_df[ts_col].min(), unit='s') #pd.to_datetime(ts, unit='ms')
#     t_max = pd.to_datetime(match_df[ts_col].max(), unit='s')

#     duration = (t_max - t_min).total_seconds()

#     print(f"\n[8] Match start (relative) : {t_min}")
#     print(f"[9] Match end   (relative): {t_max}")
#     print(f"[10] Match duration              : {duration:.1f} seconds ({duration/60:.2f} min)")

# if session_col and session_col in match_df.columns:
#     print(f"\n[11] Unique sessions in match    : {match_df[session_col].nunique()}")

# print(f"\n[12] Null counts per column:")
# print(match_df.isnull().sum())

# print(f"\n[13] Events per user:")
# print(match_df.groupby(user_col).size().sort_values(ascending=False))

# print(f"\n[14] Sample rows (head 5):")
# print(match_df.head(5))


# # ============================================================
# # BLOCK 2 — USER SANITY CHECK
# # ============================================================
# user_df = df[df[user_col] == USER_ID]

# print("\n" + "=" * 60)
# print(f"USER SANITY CHECK: {USER_ID}")
# print("=" * 60)

# print(f"\n[1] Total events for user        : {len(user_df)}")
# print(f"[2] Unique matches played        : {user_df[match_col].nunique()}")
# print(f"[3] Match IDs:")
# print(user_df[match_col].unique())

# if event_col in user_df.columns:
#     print(f"\n[4] Unique event types           : {user_df[event_col].nunique()}")
#     print("\n[5] Event distribution:")
#     print(user_df[event_col].value_counts())

# if ts_col in user_df.columns:
#     print(f"\n[6] First seen                   : {user_df[ts_col].min()}")
#     print(f"[7] Last seen                    : {user_df[ts_col].max()}")
#     # lifespan = (user_df[ts_col].max() - user_df[ts_col].min()).days
#     # print(f"[8] User lifespan                : {lifespan} days")

# if is_bot_col and is_bot_col in user_df.columns:
#     print(f"\n[9] Is bot                       : {user_df[is_bot_col].iloc[0]}")

# if session_col and session_col in user_df.columns:
#     print(f"\n[10] Unique sessions             : {user_df[session_col].nunique()}")

# print(f"\n[11] Events per match (this user):")
# print(user_df.groupby(match_col).size().sort_values(ascending=False))

# print(f"\n[12] Null counts:")
# print(user_df.isnull().sum())

# print(f"\n[13] Sample rows (head 5):")
# print(user_df.head(5))

In [ ]:
# # ============================================================
# # BLOCK 0 — OVERALL df_all SANITY CHECK
# # ============================================================

# print("=" * 60)
# print("OVERALL df_all SANITY CHECK")
# print("=" * 60)

# print(f"\n[1] Total rows                   : {len(df_all):,}")
# print(f"[2] Total columns                : {df_all.shape[1]}")
# print(f"\n[3] Column names & dtypes:")
# print(df_all.dtypes)

# print(f"\n[4] Unique matches               : {df_all[match_col].nunique():,}")
# print(f"[5] Unique users                 : {df_all[user_col].nunique():,}")

# if is_bot_col in df_all.columns:
#     bot_user_flags = df_all.groupby(user_col)[is_bot_col].first()
#     n_bots   = int(bot_user_flags.sum())
#     n_humans = int((~bot_user_flags.astype(bool)).sum())
#     print(f"\n[6] Unique human users           : {n_humans:,}")
#     print(f"[7] Unique bot users             : {n_bots:,}")
#     print(f"[8] Bot %                        : {n_bots / (n_bots + n_humans) * 100:.2f}%")
# else:
#     print("\n[6/7/8] is_bot column not found — skipping")

# if event_col in df_all.columns:
#     print(f"\n[9] Unique event types           : {df_all[event_col].nunique()}")
#     print("\n[10] Event type distribution (all):")
#     print(df_all[event_col].value_counts())

# if ts_col in df_all.columns:
#     t_min_all = pd.to_datetime(df_all[ts_col].min(), unit='s')
#     t_max_all = pd.to_datetime(df_all[ts_col].max(), unit='s')
#     span_days = (t_max_all - t_min_all).total_seconds() / 86400
#     print(f"\n[11] Earliest event              : {t_min_all}")
#     print(f"[12] Latest event                : {t_max_all}")
#     print(f"[13] Total time span             : {span_days:.2f} days")

# print(f"\n[14] Events per match (distribution):")
# epm = df_all.groupby(match_col).size()
# print(epm.describe().round(2))

# print(f"\n[15] Events per user (distribution):")
# epu = df_all.groupby(user_col).size()
# print(epu.describe().round(2))

# print(f"\n[16] Matches per user (distribution):")
# mpu = df_all.groupby(user_col)[match_col].nunique()
# print(mpu.describe().round(2))

# print(f"\n[17] Null counts per column:")
# print(df_all.isnull().sum())

# print(f"\n[18] Duplicate rows              : {df_all.duplicated().sum():,}")

# print(f"\n[19] Memory usage:")
# print(df_all.memory_usage(deep=True).sum() / 1024**2, "MB")

# print(f"\n[20] Sample rows (head 5):")
# print(df_all.head(5))